**Question-bank data not included.** Neither CSV this notebook reads is in the repository. Results are written to `outputs/`.

**Single-mask test** reads `dataset/single_mask_event_question_bank_1999_2024.csv`. `load_question_bank` requires:

- `Year` (int): the event year, compared with each model's training-cutoff year
- `Prompt` (str): a sentence with exactly one `{}` placeholder, which is replaced by the model's mask token
- `Answer` (str): the expected token for the mask
- `Theme` (str): a topic label used to group results

**Pairwise A/B test** reads `dataset/temporal_pairwise_balanced_1999_2024_natural_context.csv`. Its `load_question_bank` requires:

- `Year` (int): the event year
- `Prompt` (str): a sentence with one `{}` placeholder, filled with as many mask tokens as the candidate answer has tokens
- `A`, `B` (str): the two candidate answers
- `Correct_Option` (str): `A` or `B`
- `Category` (str): a topic label used to group results
- `Relation` (str): a relation label carried into the results

Both accept an optional `QuestionID` column; otherwise IDs `q00000`, `q00001`, ... are generated.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "outputs/question_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

QUESTION_BANK_PATH = "dataset/single_mask_event_question_bank_1999_2024.csv"


def extract_cutoff_year(model_name: str):
    """
    For ChronoBERT names like:
    manelalab/chrono-bert-v1-20151231 -> 2015
    Returns None for non-chrono models.
    """
    m = re.search(r"(\d{4})1231$", model_name)
    return int(m.group(1)) if m else None

def safe_model_name(model_name: str):
    return model_name.split("/")[-1].replace(".", "_")


def load_question_bank(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = ["Year", "Prompt", "Answer", "Theme"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "QuestionID" not in df.columns:
        df = df.copy()
        df["QuestionID"] = [f"q{i:05d}" for i in range(len(df))]

    df["Year"] = df["Year"].astype(int)
    df["Prompt"] = df["Prompt"].astype(str)
    df["Answer"] = df["Answer"].astype(str)
    df["Theme"] = df["Theme"].astype(str)

    return df.sort_values("Year").reset_index(drop=True)


def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def render_single_mask_prompt(prompt_template: str, mask_token: str) -> str:
    if "{}" not in prompt_template:
        raise ValueError("Prompt must contain exactly one '{}' placeholder.")
    if prompt_template.count("{}") != 1:
        raise ValueError("Prompt must contain exactly one '{}' placeholder.")
    return prompt_template.format(mask_token)


@torch.no_grad()
def predict_single_mask_topk(
    prompt_template,
    tokenizer,
    model,
    k=10,
    device="cpu",
    max_length=128,
):
    masked_prompt = render_single_mask_prompt(prompt_template, tokenizer.mask_token)

    enc = tokenizer(
        masked_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    mask_positions = (input_ids[0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].tolist()

    if len(mask_positions) != 1:
        raise ValueError(f"Expected exactly 1 mask token, found {len(mask_positions)}")

    mask_pos = mask_positions[0]

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[0, mask_pos]
    probs = torch.softmax(logits, dim=-1)

    topk = torch.topk(probs, k=k)
    top_ids = topk.indices.tolist()
    top_probs = topk.values.tolist()

    top_tokens = tokenizer.convert_ids_to_tokens(top_ids)
    top_decoded = [tokenizer.decode([idx]).strip() for idx in top_ids]

    preds = []
    for rank, (tok_id, tok, dec, prob) in enumerate(zip(top_ids, top_tokens, top_decoded, top_probs), start=1):
        preds.append({
            "rank": rank,
            "token_id": tok_id,
            "token": tok,
            "decoded": dec,
            "prob": float(prob),
        })

    return {
        "masked_prompt": masked_prompt,
        "topk": preds,
        "top1_token": preds[0]["token"],
        "top1_decoded": preds[0]["decoded"],
        "top1_prob": preds[0]["prob"],
    }


def evaluate_question_row(
    row,
    tokenizer,
    model,
    device="cpu",
    max_length=128,
    k=10,
):
    prompt = str(row["Prompt"])
    true_answer = str(row["Answer"])

    pred_out = predict_single_mask_topk(
        prompt_template=prompt,
        tokenizer=tokenizer,
        model=model,
        k=k,
        device=device,
        max_length=max_length,
    )

    predicted_text = pred_out["top1_decoded"]
    is_exact_match = normalize_text(predicted_text) == normalize_text(true_answer)

    topk_strings = [x["decoded"] for x in pred_out["topk"]]
    is_in_topk = normalize_text(true_answer) in [normalize_text(x) for x in topk_strings]

    return {
        "QuestionID": row["QuestionID"],
        "Year": int(row["Year"]),
        "Theme": row["Theme"],
        "Prompt": prompt,
        "Masked_Prompt": pred_out["masked_prompt"],
        "True_Answer": true_answer,
        "Predicted_Answer": predicted_text,
        "Predicted_Token": pred_out["top1_token"],
        "Top1_Prob": pred_out["top1_prob"],
        "Exact_Match": is_exact_match,
        "True_In_TopK": is_in_topk,
        "TopK_Predictions": " | ".join(topk_strings),
    }


def make_gap_bin(g):
    if g <= -6:
        return "<= -6"
    elif -5 <= g <= -1:
        return "-5 to -1"
    elif 0 <= g <= 2:
        return "0 to +2"
    elif 3 <= g <= 5:
        return "+3 to +5"
    else:
        return ">= +6"


def run_question_bank_eval(
    model_name,
    question_bank_path,
    output_dir=OUTPUT_DIR,
    device="cpu",
    max_length=128,
    k=10,
    save_plots=True,
    save_csvs=True,
    verbose=True,
):
    os.makedirs(output_dir, exist_ok=True)

    model_tag = safe_model_name(model_name)
    cutoff_year = extract_cutoff_year(model_name)

    if verbose:
        print("=" * 80)
        print("Running:", model_name)
        print("Device:", device)
        print("Cutoff year:", cutoff_year)

    # Load model
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.to(device)
    model.eval()

    # Load question bank
    df_q = load_question_bank(question_bank_path)

    # Evaluate rows
    results = []
    for _, row in tqdm(df_q.iterrows(), total=len(df_q), desc=f"Evaluating {model_tag}"):
        try:
            res = evaluate_question_row(
                row=row,
                tokenizer=tokenizer,
                model=model,
                device=device,
                max_length=max_length,
                k=k,
            )
            res["Error"] = None
        except Exception as e:
            res = {
                "QuestionID": row["QuestionID"],
                "Year": int(row["Year"]),
                "Theme": row["Theme"],
                "Prompt": row["Prompt"],
                "Masked_Prompt": None,
                "True_Answer": row["Answer"],
                "Predicted_Answer": None,
                "Predicted_Token": None,
                "Top1_Prob": np.nan,
                "Exact_Match": False,
                "True_In_TopK": False,
                "TopK_Predictions": None,
                "Error": str(e),
            }
        results.append(res)

    results_df = pd.DataFrame(results)
    valid_df = results_df[results_df["Error"].isna()].copy()

    if cutoff_year is not None:
        valid_df["Gap_To_Cutoff"] = valid_df["Year"] - cutoff_year
        valid_df["Gap_Bin"] = valid_df["Gap_To_Cutoff"].apply(make_gap_bin)

    # Overall summary
    overall_summary = {
        "model": model_name,
        "model_tag": model_tag,
        "cutoff_year": cutoff_year,
        "n_questions_total": len(df_q),
        "n_questions_scored": len(valid_df),
        "exact_match_accuracy": valid_df["Exact_Match"].mean() if len(valid_df) else np.nan,
        "topk_hit_rate": valid_df["True_In_TopK"].mean() if len(valid_df) else np.nan,
        "mean_top1_prob": valid_df["Top1_Prob"].mean() if len(valid_df) else np.nan,
    }
    overall_summary_df = pd.DataFrame([overall_summary])

    # Pre/post summary
    if cutoff_year is not None:
        pre_df = valid_df[valid_df["Year"] <= cutoff_year]
        post_df = valid_df[valid_df["Year"] > cutoff_year]

        pre_post_summary = pd.DataFrame({
            "group": ["pre_cutoff", "post_cutoff"],
            "n_questions": [len(pre_df), len(post_df)],
            "exact_match_accuracy": [
                pre_df["Exact_Match"].mean() if len(pre_df) else np.nan,
                post_df["Exact_Match"].mean() if len(post_df) else np.nan,
            ],
            "topk_hit_rate": [
                pre_df["True_In_TopK"].mean() if len(pre_df) else np.nan,
                post_df["True_In_TopK"].mean() if len(post_df) else np.nan,
            ],
            "mean_top1_prob": [
                pre_df["Top1_Prob"].mean() if len(pre_df) else np.nan,
                post_df["Top1_Prob"].mean() if len(post_df) else np.nan,
            ],
        })
    else:
        pre_post_summary = None

    # Gap summary
    if cutoff_year is not None:
        gap_summary = (
            valid_df.groupby("Gap_Bin")
            .agg(
                n_questions=("QuestionID", "count"),
                exact_match_accuracy=("Exact_Match", "mean"),
                topk_hit_rate=("True_In_TopK", "mean"),
                mean_top1_prob=("Top1_Prob", "mean"),
            )
            .reindex(["<= -6", "-5 to -1", "0 to +2", "+3 to +5", ">= +6"])
            .reset_index()
        )
    else:
        gap_summary = None

    # Year summary
    year_summary = (
        valid_df.groupby("Year")
        .agg(
            n_questions=("QuestionID", "count"),
            exact_match_accuracy=("Exact_Match", "mean"),
            topk_hit_rate=("True_In_TopK", "mean"),
            mean_top1_prob=("Top1_Prob", "mean"),
        )
        .reset_index()
        .sort_values("Year")
    )

    # Theme summary
    theme_summary = (
        valid_df.groupby("Theme")
        .agg(
            n_questions=("QuestionID", "count"),
            exact_match_accuracy=("Exact_Match", "mean"),
            topk_hit_rate=("True_In_TopK", "mean"),
            mean_top1_prob=("Top1_Prob", "mean"),
        )
        .reset_index()
        .sort_values("exact_match_accuracy", ascending=False)
    )

    # Save CSVs
    if save_csvs:
        results_df.to_csv(os.path.join(output_dir, f"results_{model_tag}.csv"), index=False)
        overall_summary_df.to_csv(os.path.join(output_dir, f"overall_summary_{model_tag}.csv"), index=False)
        year_summary.to_csv(os.path.join(output_dir, f"year_summary_{model_tag}.csv"), index=False)
        theme_summary.to_csv(os.path.join(output_dir, f"theme_summary_{model_tag}.csv"), index=False)
        if pre_post_summary is not None:
            pre_post_summary.to_csv(os.path.join(output_dir, f"pre_post_summary_{model_tag}.csv"), index=False)
        if gap_summary is not None:
            gap_summary.to_csv(os.path.join(output_dir, f"gap_summary_{model_tag}.csv"), index=False)

    
    if save_plots:
        plt.figure(figsize=(10, 5))
        plt.plot(year_summary["Year"], year_summary["exact_match_accuracy"], marker="o")
        plt.xlabel("Question year")
        plt.ylabel("Exact-match accuracy")
        plt.title(f"Exact-match accuracy by year: {model_tag}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, f"exact_match_accuracy_by_year_{model_tag}.png"),
            dpi=150,
            bbox_inches="tight",
        )
        plt.show()
        plt.close()

    # Cleanup
    del tokenizer
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if verbose:
        print("\nOverall summary")
        display(overall_summary_df)

        print("\nPredictions by year")
        display(results_df[[
            "Year", "Prompt", "True_Answer", "Predicted_Answer",
            "Exact_Match", "True_In_TopK", "TopK_Predictions"
        ]].sort_values("Year"))

        print("\nYear summary")
        display(year_summary)

        if pre_post_summary is not None:
            print("\nPre/post cutoff summary")
            display(pre_post_summary)

        print("\nTheme summary")
        display(theme_summary)

        if gap_summary is not None:
            print("\nGap summary")
            display(gap_summary)

    return {
        "results_df": results_df,
        "overall_summary_df": overall_summary_df,
        "year_summary": year_summary,
        "pre_post_summary": pre_post_summary,
        "gap_summary": gap_summary,
        "theme_summary": theme_summary,
    }


def run_multiple_models(
    model_names,
    question_bank_path,
    output_dir=OUTPUT_DIR,
    device="cpu",
    max_length=128,
    k=10,
):
    all_overall = []
    all_pre_post = []

    for model_name in model_names:
        out = run_question_bank_eval(
            model_name=model_name,
            question_bank_path=question_bank_path,
            output_dir=output_dir,
            device=device,
            max_length=max_length,
            k=k,
            save_plots=True,
            save_csvs=True,
            verbose=True,
        )

        overall_df = out["overall_summary_df"].copy()
        all_overall.append(overall_df)

        if out["pre_post_summary"] is not None:
            pp = out["pre_post_summary"].copy()
            pp["model"] = model_name
            all_pre_post.append(pp)

    overall_compare_df = pd.concat(all_overall, ignore_index=True)
    overall_compare_df = overall_compare_df.sort_values("exact_match_accuracy", ascending=False)
    overall_compare_df.to_csv(os.path.join(output_dir, "overall_compare_all_models.csv"), index=False)

    if all_pre_post:
        pre_post_compare_df = pd.concat(all_pre_post, ignore_index=True)
        pre_post_compare_df.to_csv(os.path.join(output_dir, "pre_post_compare_all_models.csv"), index=False)
    else:
        pre_post_compare_df = None

    print("\nOverall comparison across models")
    display(overall_compare_df)

    if pre_post_compare_df is not None:
        print("\nPre/post comparison across models")
        display(pre_post_compare_df)

    return overall_compare_df, pre_post_compare_df

In [ ]:
out = run_question_bank_eval(
    model_name="manelalab/chrono-bert-v1-20241231",
    question_bank_path=QUESTION_BANK_PATH,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    max_length=128,
    k=2,
    save_plots=True,
    save_csvs=True,
    verbose=True,
)

In [ ]:
MODEL_LIST = [
    "manelalab/chrono-bert-v1-20001231",
    "manelalab/chrono-bert-v1-20051231",
    "manelalab/chrono-bert-v1-20101231",
    "manelalab/chrono-bert-v1-20151231",
    "manelalab/chrono-bert-v1-20201231",
    "manelalab/chrono-bert-v1-20241231",
    "answerdotai/ModernBERT-base",
]

overall_compare_df, pre_post_compare_df = run_multiple_models(
    model_names=MODEL_LIST,
    question_bank_path=QUESTION_BANK_PATH,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    max_length=128,
    k=2,
)

## Pairwise A/B test

The model scores each of two candidate answers by the mean log-probability of the answer tokens in the masked span, and is correct when it prefers `Correct_Option`. This section redefines the helper functions above under the same names, so run the notebook top to bottom.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "outputs/pairwise_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

QUESTION_BANK_PATH = "dataset/temporal_pairwise_balanced_1999_2024_natural_context.csv"   # change this 

def extract_cutoff_year(model_name: str):
    """
    For ChronoBERT names like:
    manelalab/chrono-bert-v1-20151231 -> 2015
    Returns None for non-chrono models.
    """
    m = re.search(r"(\d{4})1231$", model_name)
    return int(m.group(1)) if m else None

def safe_model_name(model_name: str):
    return model_name.split("/")[-1].replace(".", "_")

def render_masks(n_masks: int, mask_token: str):
    return " ".join([mask_token] * n_masks)

def answer_to_token_ids(answer_text: str, tokenizer):
    enc = tokenizer(answer_text, add_special_tokens=False, return_attention_mask=False)
    token_ids = enc["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    return token_ids, tokens


@torch.no_grad()
def score_option_with_masked_answer(prompt_template, answer_text, tokenizer, model, device="cpu", max_length=128):
    """
    Score only the answer span.
    Higher mean_logp is better.
    """
    answer_token_ids, answer_tokens = answer_to_token_ids(answer_text, tokenizer)

    n_masks = len(answer_token_ids)
    masked_span = render_masks(n_masks, tokenizer.mask_token)
    masked_prompt = prompt_template.format(masked_span)

    enc = tokenizer(
        masked_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    input_ids = enc["input_ids"][0]
    attention_mask = enc["attention_mask"][0]

    mask_positions = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].tolist()

    if len(mask_positions) != n_masks:
        raise ValueError(
            f"Mask count mismatch for answer '{answer_text}'. "
            f"Expected {n_masks}, found {len(mask_positions)}."
        )

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    outputs = model(
        input_ids=input_ids.unsqueeze(0),
        attention_mask=attention_mask.unsqueeze(0),
    )

    log_probs = torch.log_softmax(outputs.logits[0], dim=-1)

    token_logps = []
    for pos, tok_id in zip(mask_positions, answer_token_ids):
        token_logps.append(log_probs[pos, tok_id].item())

    return {
        "mean_logp": float(np.mean(token_logps)),
        "sum_logp": float(np.sum(token_logps)),
        "n_answer_tokens": n_masks,
        "answer_tokens": answer_tokens,
    }


def evaluate_question_row(row, tokenizer, model, device="cpu", max_length=128):
    prompt = str(row["Prompt"])
    options = {
        "A": str(row["A"]),
        "B": str(row["B"]),
    }
    correct_option = str(row["Correct_Option"]).strip()

    scores = {}
    details = {}

    for opt, ans in options.items():
        out = score_option_with_masked_answer(
            prompt_template=prompt,
            answer_text=ans,
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_length=max_length,
        )
        scores[opt] = out["mean_logp"]
        details[opt] = out

    predicted_option = max(scores, key=scores.get)
    correct_score = scores[correct_option]
    best_wrong_score = max(v for k, v in scores.items() if k != correct_option)
    margin = correct_score - best_wrong_score

    return {
        "QuestionID": row["QuestionID"],
        "Year": int(row["Year"]),
        "Category": row["Category"],
        "Relation": row["Relation"] if "Relation" in row else None,
        "Prompt": prompt,
        "Correct_Option": correct_option,
        "Predicted_Option": predicted_option,
        "Is_Correct": predicted_option == correct_option,
        "Correct_Answer_Text": options[correct_option],
        "Predicted_Answer_Text": options[predicted_option],
        "Score_A": scores["A"],
        "Score_B": scores["B"],
        "Correct_Score": correct_score,
        "Best_Wrong_Score": best_wrong_score,
        "Margin": margin,
        "Correct_Answer_NumTokens": details[correct_option]["n_answer_tokens"],
    }


def load_question_bank(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = ["Year", "Prompt", "A", "B", "Correct_Option", "Category", "Relation"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "QuestionID" not in df.columns:
        df = df.copy()
        df["QuestionID"] = [f"q{i:05d}" for i in range(len(df))]

    df["Correct_Option"] = df["Correct_Option"].astype(str).str.strip()
    df["Category"] = df["Category"].astype(str).str.strip()
    df["Relation"] = df["Relation"].astype(str).str.strip()
    df["Year"] = df["Year"].astype(int)

    return df


def make_gap_bin(g):
    if g <= -6:
        return "<= -6"
    elif -5 <= g <= -1:
        return "-5 to -1"
    elif 0 <= g <= 2:
        return "0 to +2"
    elif 3 <= g <= 5:
        return "+3 to +5"
    else:
        return ">= +6"


def run_question_bank_eval(
    model_name,
    question_bank_path,
    output_dir=OUTPUT_DIR,
    device="cpu",
    max_length=128,
    save_plots=True,
    save_csvs=True,
    verbose=True,
):
    os.makedirs(output_dir, exist_ok=True)

    model_tag = safe_model_name(model_name)
    cutoff_year = extract_cutoff_year(model_name)

    if verbose:
        print("=" * 80)
        print("Running:", model_name)
        print("Device:", device)
        print("Cutoff year:", cutoff_year)

    # Load model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.to(device)
    model.eval()

    # Load data
    df_q = load_question_bank(question_bank_path)

    # Evaluate
    results = []
    for _, row in tqdm(df_q.iterrows(), total=len(df_q), desc=f"Evaluating {model_tag}"):
        try:
            res = evaluate_question_row(
                row=row,
                tokenizer=tokenizer,
                model=model,
                device=device,
                max_length=max_length,
            )
            res["Error"] = None
        except Exception as e:
            res = {
                "QuestionID": row["QuestionID"],
                "Year": int(row["Year"]),
                "Category": row["Category"],
                "Relation": row["Relation"] if "Relation" in row else None,
                "Prompt": row["Prompt"],
                "Correct_Option": row["Correct_Option"],
                "Predicted_Option": None,
                "Is_Correct": False,
                "Correct_Answer_Text": None,
                "Predicted_Answer_Text": None,
                "Score_A": np.nan,
                "Score_B": np.nan,
                "Correct_Score": np.nan,
                "Best_Wrong_Score": np.nan,
                "Margin": np.nan,
                "Correct_Answer_NumTokens": np.nan,
                "Error": str(e),
            }
        results.append(res)

    results_df = pd.DataFrame(results)
    valid_df = results_df.dropna(subset=["Correct_Score"]).copy()

    if cutoff_year is not None:
        valid_df["Gap_To_Cutoff"] = valid_df["Year"] - cutoff_year
        valid_df["Gap_Bin"] = valid_df["Gap_To_Cutoff"].apply(make_gap_bin)

    # Overall summary
    overall_summary = {
        "model": model_name,
        "model_tag": model_tag,
        "cutoff_year": cutoff_year,
        "n_questions_total": len(df_q),
        "n_questions_scored": len(valid_df),
        "overall_accuracy": valid_df["Is_Correct"].mean(),
        "mean_correct_score": valid_df["Correct_Score"].mean(),
        "mean_margin": valid_df["Margin"].mean(),
    }
    overall_summary_df = pd.DataFrame([overall_summary])

    # Pre/post summary
    if cutoff_year is not None:
        pre_df = valid_df[valid_df["Year"] <= cutoff_year]
        post_df = valid_df[valid_df["Year"] > cutoff_year]

        pre_post_summary = pd.DataFrame({
            "group": ["pre_cutoff", "post_cutoff"],
            "n_questions": [len(pre_df), len(post_df)],
            "accuracy": [pre_df["Is_Correct"].mean(), post_df["Is_Correct"].mean()],
            "mean_correct_score": [pre_df["Correct_Score"].mean(), post_df["Correct_Score"].mean()],
            "mean_margin": [pre_df["Margin"].mean(), post_df["Margin"].mean()],
        })
    else:
        pre_post_summary = None

    # Gap summary
    if cutoff_year is not None:
        gap_summary = (
            valid_df.groupby("Gap_Bin")
            .agg(
                n_questions=("QuestionID", "count"),
                accuracy=("Is_Correct", "mean"),
                mean_correct_score=("Correct_Score", "mean"),
                mean_margin=("Margin", "mean"),
            )
            .reindex(["<= -6", "-5 to -1", "0 to +2", "+3 to +5", ">= +6"])
            .reset_index()
        )
    else:
        gap_summary = None

    # Year summary
    year_summary = (
        valid_df.groupby("Year")
        .agg(
            n_questions=("QuestionID", "count"),
            accuracy=("Is_Correct", "mean"),
            mean_correct_score=("Correct_Score", "mean"),
            mean_margin=("Margin", "mean"),
        )
        .reset_index()
        .sort_values("Year")
    )

    # Category summary
    category_summary = (
        valid_df.groupby("Category")
        .agg(
            n_questions=("QuestionID", "count"),
            accuracy=("Is_Correct", "mean"),
            mean_correct_score=("Correct_Score", "mean"),
            mean_margin=("Margin", "mean"),
        )
        .reset_index()
        .sort_values("accuracy", ascending=False)
    )

    # Save CSVs
    if save_csvs:
        results_df.to_csv(os.path.join(output_dir, f"results_{model_tag}.csv"), index=False)
        overall_summary_df.to_csv(os.path.join(output_dir, f"overall_summary_{model_tag}.csv"), index=False)
        year_summary.to_csv(os.path.join(output_dir, f"year_summary_{model_tag}.csv"), index=False)
        category_summary.to_csv(os.path.join(output_dir, f"category_summary_{model_tag}.csv"), index=False)
        if pre_post_summary is not None:
            pre_post_summary.to_csv(os.path.join(output_dir, f"pre_post_summary_{model_tag}.csv"), index=False)
        if gap_summary is not None:
            gap_summary.to_csv(os.path.join(output_dir, f"gap_summary_{model_tag}.csv"), index=False)

    # Plot: ONLY accuracy by year
    if save_plots:
        plt.figure(figsize=(10, 5))
        plt.plot(year_summary["Year"], year_summary["accuracy"], marker="o")
        plt.xlabel("Question year")
        plt.ylabel("Accuracy")
        plt.title(f"Accuracy by year: {model_tag}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, f"accuracy_by_year_{model_tag}.png"),
            dpi=150,
            bbox_inches="tight"
        )
        plt.show()

    # Cleanup model memory
    del tokenizer
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if verbose:
        print("\nOverall summary")
        display(overall_summary_df)

        print("\nYear summary")
        display(year_summary)

        if pre_post_summary is not None:
            print("\nPre/post cutoff summary")
            display(pre_post_summary)

        print("\nCategory summary")
        display(category_summary)

        if gap_summary is not None:
            print("\nGap summary")
            display(gap_summary)

    return {
        "results_df": results_df,
        "overall_summary_df": overall_summary_df,
        "year_summary": year_summary,
        "pre_post_summary": pre_post_summary,
        "gap_summary": gap_summary,
        "category_summary": category_summary,
    }


def run_multiple_models(
    model_names,
    question_bank_path,
    output_dir=OUTPUT_DIR,
    device="cpu",
    max_length=128,
):
    all_overall = []
    all_pre_post = []

    for model_name in model_names:
        out = run_question_bank_eval(
            model_name=model_name,
            question_bank_path=question_bank_path,
            output_dir=output_dir,
            device=device,
            max_length=max_length,
            save_plots=True,
            save_csvs=True,
            verbose=True,
        )

        overall_df = out["overall_summary_df"].copy()
        all_overall.append(overall_df)

        if out["pre_post_summary"] is not None:
            pp = out["pre_post_summary"].copy()
            pp["model"] = model_name
            all_pre_post.append(pp)

    overall_compare_df = pd.concat(all_overall, ignore_index=True)
    overall_compare_df = overall_compare_df.sort_values("overall_accuracy", ascending=False)

    overall_compare_path = os.path.join(output_dir, "overall_compare_all_models.csv")
    overall_compare_df.to_csv(overall_compare_path, index=False)

    if all_pre_post:
        pre_post_compare_df = pd.concat(all_pre_post, ignore_index=True)
        pre_post_compare_path = os.path.join(output_dir, "pre_post_compare_all_models.csv")
        pre_post_compare_df.to_csv(pre_post_compare_path, index=False)
    else:
        pre_post_compare_df = None

    print("\nOverall comparison across models")
    display(overall_compare_df)

    if pre_post_compare_df is not None:
        print("\nPre/post comparison across models")
        display(pre_post_compare_df)

    return overall_compare_df, pre_post_compare_df

In [ ]:
single_out = run_question_bank_eval(
    model_name="answerdotai/ModernBERT-large",
    question_bank_path=QUESTION_BANK_PATH,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    save_csvs=False,
    save_plots=True,
    max_length=128,
)

In [ ]:
MODEL_LIST = [
    "manelalab/chrono-bert-v1-20001231",
    "manelalab/chrono-bert-v1-20051231",
    "manelalab/chrono-bert-v1-20101231",
    "manelalab/chrono-bert-v1-20151231",
    "manelalab/chrono-bert-v1-20201231",
    "manelalab/chrono-bert-v1-20241231",
    "answerdotai/ModernBERT-base",
]

overall_compare_df, pre_post_compare_df = run_multiple_models(
    model_names=MODEL_LIST,
    question_bank_path=QUESTION_BANK_PATH,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    max_length=128,
)